In [49]:
from dotenv import load_dotenv
import os
import re

In [50]:
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

In [51]:
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
import json
import os

In [52]:
# Modelo que quieres usar
MODEL = "gemini-2.5-flash-lite"

In [53]:
# Configuración del modelo Gemini
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.7, google_api_key=API_KEY)

In [54]:
# Preprocesador: limpia espacios y limita a 500 caracteres
def preprocess_text(text):
    """Limpia el texto eliminando espacios extras y limitando longitud"""
    return text.strip()[:500]

preprocessor = RunnableLambda(preprocess_text)

# Generación de resumen
def generate_summary(text):
    """Genera un resumen conciso del texto"""
    prompt = f"Resume en una sola oración el siguiente texto:\n{text}"
    response = llm.invoke(prompt)
    return response.content

summary_branch = RunnableLambda(generate_summary)

# Análisis de sentimiento con formato JSON
def analyze_sentiment(text):
    """Analiza el sentimiento y devuelve resultado estructurado"""
    prompt = f"""
Analiza el sentimiento del siguiente texto.
Responde ÚNICAMENTE en formato JSON válido con esta estructura:
{{"sentimiento": "positivo|negativo|neutro", "razon": "justificación breve"}}

Texto:
{text}
"""
    response = llm.invoke(prompt)
    # Eliminar posibles ```json ```
    try:
        return json.loads(response.content)
    except json.JSONDecodeError:
        return {
            "sentimiento": "neutro",
            "razon": "No se pudo parsear la respuesta del modelo"
        }

sentiment_branch = RunnableLambda(analyze_sentiment)

# Combinación de resultados
def merge_results(data):
    """Combina los resultados de ambas ramas en un formato unificado"""
    return {
        "resumen": data["resumen"],
        "sentimiento": data["sentimiento_data"]["sentimiento"],
        "razon": data["sentimiento_data"]["razon"]
    }

merger = RunnableLambda(merge_results)

In [55]:
# Ejecución en paralelo
parallel_analysis = RunnableParallel({
    "resumen": summary_branch,
    "sentimiento_data": sentiment_branch
})

# Cadena completa
chain = preprocessor | parallel_analysis | merger

In [ ]:
reviews_batch = [
    "El envío fue rápido y el artículo llegó en perfectas condiciones",
    "La experiencia fue decepcionante, el producto no funciona como esperaba"
]

resultado_batch = chain.batch(reviews_batch)

print(resultado_batch)